# Agente de Mantenimiento Eléctrico SH-32

**Ingeniería de Soluciones con Inteligencia Artificial (ISY0101) · Evaluación Parcial N°2**

**Integrantes:** Diego Cabrera, Luis Muñoz, Aran Opazo, Eric Rubilar

Un taller de mantenimiento recibe reportes de fallas eléctricas de la flota de helicópteros SH-32. Hoy una persona busca en el manual, revisa si la falla ya ocurrió, decide qué tan grave es, consulta la bodega, abre una orden de trabajo (OT) y avisa a quien corresponde.

En este notebook ese trabajo lo hace un equipo de agentes con CrewAI. Un Manager (Jefe de Taller) recibe cada solicitud y reparte el trabajo entre tres especialistas. La búsqueda en el manual que hicimos en la primera parte del proyecto ahora es una de sus herramientas. Las fallas graves no se aprueban solas: quedan esperando a una persona.

**Contenido**

0. Instalación y configuración
1. Búsqueda en el manual y umbral de relevancia
2. Entorno simulado del taller
3. Memoria
4. Herramientas y pruebas
5. Equipo de agentes
6. Escenarios de prueba y resumen
7. Continuidad entre sesiones
8. Evidencias para el repositorio
9. Uso libre para la demo

## Arquitectura

Cada solicitud pasa primero por la memoria. Luego el Manager decide qué especialistas necesita y en qué orden. Al final el código revisa la orden de trabajo, envía los correos que falten y guarda un registro de todo lo que pasó.

## 0. Instalación y configuración

La celda siguiente instala los paquetes (solo si faltan, con versiones fijas) y reúne **todos los imports** del notebook. Si aparece un error de versiones, usa **Entorno de ejecución > Reiniciar sesión** y vuelve a ejecutar esa misma celda: si ya está todo instalado, la instalación se omite.

In [ ]:
# Instala lo que falte y carga las librerías
import sys, subprocess
from pathlib import Path
import importlib.metadata as _md
import sys, subprocess, os

EN_COLAB = "google.colab" in sys.modules

# Versiones fijas para que la instalación sea rápida y estable
VERSIONES = {
    "crewai": "1.6.1",
    "litellm": "1.106.2",
    "langchain-community": "0.3.27",
    "langchain-text-splitters": "0.3.11",
    "langchain-huggingface": "0.3.1",
    "faiss-cpu": "1.15.1",
    "pdfplumber": "0.11.10",
}

def _version(paquete):
    try:
        return _md.version(paquete)
    except _md.PackageNotFoundError:
        return None

_faltan = [p for p, v in VERSIONES.items() if _version(p) != v]
if EN_COLAB and _faltan:
    print("Instalando:", ", ".join(_faltan), "(tarda unos minutos)...")
    Path("constraints_colab.txt").write_text(
        "requests==2.32.4\n"
        "oauthlib>=3.2.0,<4\n"
        "pillow>=8,<13\n"
        "rich>=12.4.4,<14\n"
        "opentelemetry-api>=1.39,<=1.42.1\n"
        "opentelemetry-sdk>=1.39,<=1.42.1\n",
        encoding="utf-8",
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", "constraints_colab.txt",
                           *[f"{p}=={v}" for p, v in VERSIONES.items()]])
    # Se reinicia la sesión para cargar las versiones nuevas
    print("Instalación lista. La sesión se reinicia sola: vuelve a usar 'Ejecutar todo'.")
    os.kill(os.getpid(), 9)
else:
    print("Paquetes ya instalados.")

# Librerías
import os, json, time, re, shutil, tempfile, warnings
from datetime import datetime
from typing import Type
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

import pandas as pd
import pdfplumber
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Deben definirse antes de importar CrewAI
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["CREWAI_TESTING"] = "true"   # evita una pregunta de CrewAI que espera 20 s

import litellm
from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import BaseTool
try:
    from crewai.events import crewai_event_bus
    from crewai.events.types.tool_usage_events import ToolUsageStartedEvent
except ImportError:
    crewai_event_bus = None

litellm.suppress_debug_info = True

if EN_COLAB:
    from google.colab import userdata, files, drive
else:
    from dotenv import load_dotenv

print("Imports listos. CrewAI", _version("crewai"), "· LiteLLM", _version("litellm"))

Paquetes ya instalados.
Imports listos. CrewAI 1.6.1 · LiteLLM 1.106.2


### 0.1 Credenciales

La API key se lee desde el panel **Secrets** de Colab. Solo `LLM_API_KEY` es obligatoria; las otras dos sirven para cambiar de proveedor.

| Secret | Groq (por defecto) | Mistral |
|---|---|---|
| `LLM_API_KEY` | key de Groq | key de Mistral |
| `LLM_BASE_URL` | `https://api.groq.com/openai/v1` | `https://api.mistral.ai/v1` |
| `LLM_MODEL` | `qwen/qwen3.8-27b` | `mistral-small-latest` |

In [ ]:
if EN_COLAB:
    # Borra la key anterior de esta sesión
    for _k in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL"):
        os.environ.pop(_k, None)
    for _k in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v.strip()
        except Exception as e:
            if type(e).__name__ == "NotebookAccessError":
                print(f"El Secret {_k} existe, pero no tiene activado el acceso a este notebook.")
            elif _k == "LLM_API_KEY":
                print(f"No se pudo leer el Secret LLM_API_KEY ({type(e).__name__}).")
else:
    load_dotenv()

os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "qwen/qwen3.8-27b")

_key = os.getenv("LLM_API_KEY", "")
print("Base URL :", os.getenv("LLM_BASE_URL"))
print("Modelo   :", os.getenv("LLM_MODEL"))
print("API key  :", "configurada" if _key else "FALTA",
      f"(empieza con '{_key[:4]}', termina en '{_key[-4:]}', largo {len(_key)})" if _key else "")

Base URL : https://api.groq.com/openai/v1
Modelo   : qwen/qwen3.8-27b
API key  : configurada (empieza con 'gsk_', termina en '9d2t', largo 56)


### 0.2 Parámetros

- `USAR_DRIVE = True` guarda el índice del manual, la memoria y las evidencias en Google Drive, para no perderlos al cerrar Colab.
- `PAUSA_ENTRE_ESCENARIOS` son los segundos de espera entre escenarios, para no pasar el límite de tokens por minuto de la cuenta gratuita.

In [ ]:
USAR_DRIVE = False   # True: guarda en Google Drive
PAUSA_ENTRE_ESCENARIOS = 30   # segundos entre escenarios

CORREO_SUPERVISION = "supervision.mantenimiento@taller-sh32.cl"
CORREO_BODEGA      = "abastecimiento@taller-sh32.cl"

if USAR_DRIVE and EN_COLAB:
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/EP2_SH32")
else:
    BASE = Path("EP2_SH32")

DIR_MANUALES  = BASE / "manuales"
DIR_INDICE    = BASE / "indice_manual"
DIR_MEMORIA   = BASE / "memoria_lp"
DIR_EVIDENCIA = BASE / "evidencias"
for d in (DIR_MANUALES, DIR_INDICE, DIR_MEMORIA, DIR_EVIDENCIA):
    d.mkdir(parents=True, exist_ok=True)

def pausa():
    if PAUSA_ENTRE_ESCENARIOS:
        print(f"Pausa de {PAUSA_ENTRE_ESCENARIOS} segundos (límite de tokens por minuto).")
        time.sleep(PAUSA_ENTRE_ESCENARIOS)

print("Carpeta de trabajo:", BASE.resolve())

Carpeta de trabajo: /content/EP2_SH32


### 0.3 Modelo

Los agentes usan CrewAI. LangChain solo se usa para la búsqueda en el manual (FAISS y embeddings).

Con Groq se agrega `reasoning_effort="none"`, porque el modelo qwen "piensa" antes de responder y gasta el límite de tokens. Colab ya tiene un *event loop* activo y CrewAI no ejecuta `kickoff()` dentro de él, por eso las llamadas se hacen en un hilo aparte con `en_hilo()`.

**Cuota:** la cuenta gratuita de Groq permite 200.000 tokens al día con qwen y el Manager hace varias llamadas por solicitud. Si se acaba, se puede usar una key de Mistral.

In [ ]:
ES_GROQ = "groq" in os.getenv("LLM_BASE_URL", "")
# En qwen se desactiva el modo de razonamiento para gastar menos tokens
_extra = {"reasoning_effort": "none"} if ES_GROQ and "qwen" in os.getenv("LLM_MODEL", "").lower() else {}

def crear_llm_crewai(max_tokens=600, temperature=0.2):
    return LLM(
        model="openai/" + os.getenv("LLM_MODEL"),
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        temperature=temperature,
        max_tokens=max_tokens,
        **_extra,
    )

def en_hilo(fn, *args, **kwargs):
    # Colab ya tiene un event loop, por eso CrewAI corre en otro hilo
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(fn, *args, **kwargs).result()

_PENSAMIENTO = re.compile(r"<think>.*?</think>", re.S | re.I)
def limpiar(texto) -> str:
    """Quita el bloque <think> si el modelo lo devuelve."""
    return _PENSAMIENTO.sub("", str(texto)).strip()

def probar_modelo(intentos: int = 3, espera: int = 20) -> bool:
    llm = crear_llm_crewai(max_tokens=20, temperature=0)
    for i in range(1, intentos + 1):
        try:
            print("Prueba del modelo:", limpiar(en_hilo(llm.call, "Responde solo: funcionando"))[:60])
            return True
        except Exception as e:
            txt = str(e).lower()
            if "401" in txt or "invalid api key" in txt or "unauthorized" in txt:
                print("El proveedor rechazó la API key (401). Revisa el Secret LLM_API_KEY y vuelve a ejecutar 0.1.")
                return False
            if "per day" in txt or "tpd" in txt:
                print("Se agotó la cuota DIARIA del proveedor. Espera a que se renueve o usa otra cuenta/modelo.")
                return False
            if ("429" in txt or "rate" in txt) and i < intentos:
                print(f"Límite de solicitudes (429). Reintento {i}/{intentos - 1} en {espera} s...")
                time.sleep(espera)
                continue
            print("El modelo no respondió:", type(e).__name__, str(e)[:300])
            return False
    return False

MODELO_OK = probar_modelo()

Prueba del modelo: funcionando


## 1. Búsqueda en el manual

Al ejecutar la celda siguiente aparece el botón **Elegir archivos** para subir `Manual electrico SH-32.pdf`. Si subes un PDF, reemplaza al anterior. Si cancelas y ya había uno guardado, se usa ese.

Cada fragmento guarda su número de página para poder citarlo. El índice se guarda en disco y solo se vuelve a calcular si cambia el PDF.

In [ ]:
if EN_COLAB:
    print('Presiona "Elegir archivos" y selecciona el PDF del manual.')
    print("Si cancelas, se usa el manual que ya esté guardado en", DIR_MANUALES)
    subidos = files.upload()
    pdf_subidos = {n: c for n, c in subidos.items() if n.lower().endswith(".pdf")}
    if pdf_subidos:
        for viejo in DIR_MANUALES.glob("*.pdf"):   # el PDF nuevo reemplaza al anterior
            viejo.unlink()
        for nombre, contenido in pdf_subidos.items():
            (DIR_MANUALES / nombre).write_bytes(contenido)
        print("Manual cargado:", ", ".join(pdf_subidos))
else:
    print(f"Fuera de Colab: copia el PDF del manual en {DIR_MANUALES.resolve()}")

pdfs = sorted(DIR_MANUALES.glob("*.pdf"))
if not pdfs:
    raise ValueError("No hay PDF del manual. Ejecuta de nuevo esta celda y súbelo.")
print("Manuales que se usarán:", [p.name for p in pdfs])

Presiona "Elegir archivos" y selecciona el PDF del manual.
Si cancelas, se usa el manual que ya esté guardado en EP2_SH32/manuales


Saving Manual electrico SH-32.pdf to Manual electrico SH-32 (1).pdf
Manual cargado: Manual electrico SH-32 (1).pdf
Manuales que se usarán: ['Manual electrico SH-32 (1).pdf']


In [ ]:
# Con embeddings normalizados, la distancia de FAISS equivale a la similitud coseno
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)

def indexar_manuales(rutas):
    paginas = []
    for ruta in rutas:
        with pdfplumber.open(ruta) as pdf:
            for i, page in enumerate(pdf.pages, start=1):
                texto = page.extract_text() or ""
                if texto.strip():
                    paginas.append(Document(page_content=texto,
                                            metadata={"fuente": ruta.name, "pagina": i}))
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(paginas)
    print(f"{len(paginas)} páginas, {len(chunks)} fragmentos")
    return FAISS.from_documents(chunks, embeddings)

firma = "|".join(f"{p.name}:{p.stat().st_size}" for p in pdfs)   # detecta si cambió el PDF
archivo_firma = DIR_INDICE / "firma.txt"
if archivo_firma.exists() and archivo_firma.read_text() == firma:
    vector_manual = FAISS.load_local(str(DIR_INDICE), embeddings,
                                     allow_dangerous_deserialization=True)
    print("Índice del manual cargado desde disco.")
else:
    vector_manual = indexar_manuales(pdfs)
    vector_manual.save_local(str(DIR_INDICE))
    archivo_firma.write_text(firma)
    print("Índice del manual creado y guardado.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

54 páginas, 146 fragmentos
Índice del manual creado y guardado.


### 1.1 Umbral de relevancia

Sirve para saber cuándo una pregunta no tiene relación con el manual. Se buscan algunas preguntas del manual y otras sin relación, y se mira la distancia del fragmento más parecido. En esta medida, **menos distancia significa más parecido**.

- De las preguntas del manual se toma la distancia más alta (la menos parecida). En la ejecución del 21/10 fue 0,994.
- De las preguntas sin relación se toma la distancia más baja (la más parecida). En esa ejecución fue 1,529.
- El umbral es el punto medio entre las dos: (0,994 + 1,529) / 2 = **1,261**.

Si el fragmento más cercano supera el umbral, `buscar_manual` responde `SIN_RESULTADOS_RELEVANTES` y el agente no inventa una respuesta. Si los dos grupos se cruzaran, se usa 1,3. La celda imprime el cálculo con los números de esta ejecución y guarda la tabla.

In [ ]:
pruebas_umbral = [
    ("¿Qué función cumple el contactor interbarras?",           "dominio"),
    ("¿Qué ocurre si se pierden los dos alternadores?",          "dominio"),
    ("¿Qué alimenta la batería de 24 V?",                        "dominio"),
    ("¿Cómo se rearma el sistema después de un cortocircuito?",  "dominio"),
    ("¿Cuántos perros tienen el pelaje blanco?",                 "fuera"),
    ("Receta de pastel de choclo",                               "fuera"),
    ("¿Quién ganó el mundial de fútbol 2010?",                   "fuera"),
]
filas = []
for q, tipo in pruebas_umbral:
    doc, dist = vector_manual.similarity_search_with_score(q, k=1)[0]
    filas.append({"pregunta": q, "tipo": tipo, "distancia": round(float(dist), 3),
                  "página": doc.metadata.get("pagina")})
df_umbral = pd.DataFrame(filas)
display(df_umbral)

max_dom   = df_umbral[df_umbral.tipo == "dominio"].distancia.max()
min_fuera = df_umbral[df_umbral.tipo == "fuera"].distancia.min()
UMBRAL_RELEVANCIA = round((max_dom + min_fuera) / 2, 3) if max_dom < min_fuera else 1.3
print(f"Máx. distancia DENTRO del dominio (la menos parecida del manual): {max_dom}")
print(f"Mín. distancia FUERA del dominio (la más parecida de lo ajeno)   : {min_fuera}")
if max_dom < min_fuera:
    print(f"UMBRAL = ({max_dom} + {min_fuera}) / 2 = {UMBRAL_RELEVANCIA}")
else:
    print(f"Los grupos se cruzan, se usa el umbral por defecto: {UMBRAL_RELEVANCIA}")
df_umbral.to_csv(DIR_EVIDENCIA / "calibracion_umbral.csv", index=False)

,pregunta,tipo,distancia,página
0,¿Qué función cumple el contactor interbarras?,dominio,0.994,18
1,¿Qué ocurre si se pierden los dos alternadores?,dominio,0.760,13
2,¿Qué alimenta la batería de 24 V?,dominio,0.489,5
3,¿Cómo se rearma el sistema después de un corto...,dominio,0.900,12
4,¿Cuántos perros tienen el pelaje blanco?,fuera,1.568,15
5,Receta de pastel de choclo,fuera,1.529,27
6,¿Quién ganó el mundial de fútbol 2010?,fuera,1.657,38


Máx. distancia DENTRO del dominio (la menos parecida del manual): 0.994
Mín. distancia FUERA del dominio (la más parecida de lo ajeno)   : 1.529
UMBRAL = (0.994 + 1.529) / 2 = 1.261


In [ ]:
def buscar_en_manual(consulta: str, k: int = 4) -> str:
    """Busca en el manual y devuelve los fragmentos con su página."""
    resultados = vector_manual.similarity_search_with_score(consulta, k=k)
    if not resultados or resultados[0][1] > UMBRAL_RELEVANCIA:
        return ("SIN_RESULTADOS_RELEVANTES: el manual eléctrico SH-32 no contiene "
                "información sobre esta consulta.")
    bloques = []
    for doc, dist in resultados:
        if dist <= UMBRAL_RELEVANCIA * 1.15:
            bloques.append(f"[{doc.metadata['fuente']} · pág. {doc.metadata['pagina']} · "
                           f"dist {dist:.2f}]\n{doc.page_content[:900]}")
    return "\n\n---\n\n".join(bloques)

print(buscar_en_manual("¿Qué función cumple el contactor interbarras?"))#[:1200])

[Manual electrico SH-32 (1).pdf · pág. 18 · dist 0.99]
THM
T1
8.5.2. LAS FUNCIONES (SIMPLIFICADAS) DE LA GENERACIÓN CONTINUA (continuación)
( 3 ) La alimentación por APU
La corriente de mando del contactor de
toma de energía exterior (33P3) viene del
APU. Cuando el interruptor (1) «BAT-EXT
PWR» está en «ON», el APU acopla en
cuanto se enchufa el APU bajo tensión.
La conexión del APU provoca:
- el cierre del contactor interbarras (33P5)
Cierre interbarras
- la prohibición del cierre del contactor
«+BAT» (33P4)
APU
Protección contra sobretensión: si la conectado Prohibición
conexión
tensión del grupo excede 32,5 V, la tarjeta
+ BAT
• TR no conectados
de protección (A) situada en el corazón
• APU energizado
eléctrico provoca la abertura del contactor.
Conexión Protección
Para rearmar, hay que desenchufar y volver contra
Desconexión
a enchufar el APU. (Desenchufando el sobretensiones
• U > 32.5 V
APU, se corta el + de mando).
P. PARC 1
Potencia 

---

[Manual electrico SH-32 (1).pdf · pág.

## 2. Entorno simulado del taller

El agente necesita sistemas sobre los cuales actuar. Se simulan tres:

| Sistema | Uso | Dónde se guarda |
|---|---|---|
| Inventario de bodega | lectura | `INVENTARIO` |
| Órdenes de trabajo (OT) | escritura | `evidencias/ordenes_trabajo.json` |
| Correo | escritura | `evidencias/correos_enviados.json` |

Los códigos de los repuestos (33P5, TR1, TR2, 49P...) salen del manual, pero el stock, las ubicaciones y los plazos son inventados. Con `ENTORNO` se puede cambiar una condición durante la ejecución, por ejemplo dejar un repuesto sin stock.

In [ ]:
INVENTARIO = {
    "33P5":    {"descripcion": "Contactor interbarras (acopla barras PP2-PP3)", "stock": 2,
                "ubicacion": "Bodega A-12", "dias_reposicion": 5,  "alternativa": "33P5-R (reacondicionado)"},
    "TR1":     {"descripcion": "Transformador-rectificador 28 V CC (barra PP3)", "stock": 1,
                "ubicacion": "Bodega B-03", "dias_reposicion": 15, "alternativa": None},
    "TR2":     {"descripcion": "Transformador-rectificador 28 V CC (barra esencial PP2)", "stock": 1,
                "ubicacion": "Bodega B-04", "dias_reposicion": 15, "alternativa": None},
    "49P":     {"descripcion": "Relé de transferencia red 1", "stock": 3,
                "ubicacion": "Bodega A-07", "dias_reposicion": 3,  "alternativa": None},
    "58X":     {"descripcion": "Relé de transferencia red 2", "stock": 1,
                "ubicacion": "Bodega A-08", "dias_reposicion": 3,  "alternativa": None},
    "35X1":    {"descripcion": "Contactor de línea alternador 1", "stock": 2,
                "ubicacion": "Bodega A-15", "dias_reposicion": 7,  "alternativa": None},
    "35X2":    {"descripcion": "Contactor de línea alternador 2", "stock": 2,
                "ubicacion": "Bodega A-16", "dias_reposicion": 7,  "alternativa": None},
    "37X1":    {"descripcion": "Transformador de intensidad red 1", "stock": 2,
                "ubicacion": "Bodega C-01", "dias_reposicion": 10, "alternativa": None},
    "BAT-24V": {"descripcion": "Batería 24 V (barra esencial PP2)", "stock": 1,
                "ubicacion": "Sala de baterías", "dias_reposicion": 4, "alternativa": None},
}
ENTORNO = {"inventario": INVENTARIO}

ARCH_OT      = DIR_EVIDENCIA / "ordenes_trabajo.json"
ARCH_CORREOS = DIR_EVIDENCIA / "correos_enviados.json"
ARCH_TRAZAS  = DIR_EVIDENCIA / "trazas_agente.jsonl"

def _leer_json(ruta):
    return json.loads(ruta.read_text(encoding="utf-8")) if ruta.exists() else []

def _guardar_json(ruta, datos):
    ruta.write_text(json.dumps(datos, ensure_ascii=False, indent=2), encoding="utf-8")

pd.DataFrame(INVENTARIO).T

,descripcion,stock,ubicacion,dias_reposicion,alternativa
33P5,Contactor interbarras (acopla barras PP2-PP3),2,Bodega A-12,5,33P5-R (reacondicionado)
TR1,Transformador-rectificador 28 V CC (barra PP3),1,Bodega B-03,15,None
TR2,Transformador-rectificador 28 V CC (barra esen...,1,Bodega B-04,15,None
49P,Relé de transferencia red 1,3,Bodega A-07,3,None
58X,Relé de transferencia red 2,1,Bodega A-08,3,None
35X1,Contactor de línea alternador 1,2,Bodega A-15,7,None
35X2,Contactor de línea alternador 2,2,Bodega A-16,7,None
37X1,Transformador de intensidad red 1,2,Bodega C-01,10,None
BAT-24V,Batería 24 V (barra esencial PP2),1,Sala de baterías,4,None


## 3. Memoria

| Tipo | Qué guarda | Cómo se usa |
|---|---|---|
| Corto plazo (`MemoriaCortoPlazo`) | Las últimas 6 solicitudes y respuestas de la sesión | Se le entrega al Manager en cada solicitud |
| Largo plazo (`MemoriaLargoPlazo`) | El historial de fallas y cada OT creada (FAISS + archivo JSONL) | Antes de cada solicitud se buscan casos parecidos; también la usa `buscar_historial_fallas` |
| Manual | Los fragmentos del manual | Herramienta `buscar_manual` |

La memoria de corto plazo es una ventana simple. La memoria con resumen de LangChain era parte del agente que dejamos de usar, y la memoria de CrewAI pide una key de OpenAI. Lo que sale de la ventana no se pierde, porque cada OT queda en la memoria de largo plazo.

In [ ]:
class MemoriaLargoPlazo:
    """Guarda cada caso en un archivo JSONL y en FAISS para buscarlo después."""

    def __init__(self, carpeta: Path, embeddings):
        self.carpeta, self.emb = carpeta, embeddings
        self.registro = carpeta / "episodios.jsonl"
        if (carpeta / "index.faiss").exists():
            self.vs = FAISS.load_local(str(carpeta), embeddings,
                                       allow_dangerous_deserialization=True)
        else:
            self.vs = None

    def recordar(self, texto: str, **metadata):
        metadata.setdefault("fecha", datetime.now().strftime("%Y-%m-%d %H:%M"))
        doc = Document(page_content=texto, metadata=metadata)
        if self.vs is None:
            self.vs = FAISS.from_documents([doc], self.emb)
        else:
            self.vs.add_documents([doc])
        self.vs.save_local(str(self.carpeta))
        with self.registro.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"texto": texto, **metadata}, ensure_ascii=False) + "\n")

    def buscar(self, consulta: str, k: int = 3, umbral: float = 1.25):
        if self.vs is None:
            return []
        return [(d, s) for d, s in self.vs.similarity_search_with_score(consulta, k=k) if s <= umbral]

    def total(self):
        return len(self.vs.index_to_docstore_id) if self.vs else 0


memoria_lp = MemoriaLargoPlazo(DIR_MEMORIA, embeddings)

# Historial simulado del taller (solo la primera vez)
if memoria_lp.total() == 0:
    historial_simulado = [
        ("SH-32-01 | Luz de transferencia encendida en vuelo; pérdida de tensión en red 1. "
         "Se reemplazó el relé de transferencia 49P. Criticidad MEDIA. OT-20260314-001.", "2026-03-14"),
        ("SH-32-02 | TR2 sin salida de 28 V; la barra esencial PP2 quedó alimentada solo por batería. "
         "Se reemplazó TR2. Criticidad ALTA, aprobada por supervisión. OT-20260502-004.", "2026-05-02"),
        ("SH-32-01 | Disparo repetido de breaker térmico en distribución 115 V monofásica; "
         "conexión floja corregida, sin cambio de repuesto. Criticidad BAJA. OT-20260721-002.", "2026-07-21"),
        ("SH-32-03 | Contactor interbarras 33P5 no rearma tras prueba de cortocircuito; se resolvió "
         "con interruptor coupling en reset y luego ON. Criticidad MEDIA. OT-20260830-003.", "2026-08-30"),
    ]
    for texto, fecha in historial_simulado:
        memoria_lp.recordar(texto, tipo="historial", fecha=fecha)
print("Episodios en memoria de largo plazo:", memoria_lp.total())

Episodios en memoria de largo plazo: 4


In [ ]:
class MemoriaCortoPlazo:

    def __init__(self, ventana: int = 6, max_chars_respuesta: int = 600):
        self.ventana, self.max_chars = ventana, max_chars_respuesta
        self.turnos = []

    def guardar(self, solicitud: str, respuesta: str):
        self.turnos.append({"hora": datetime.now().strftime("%H:%M"),
                            "solicitud": solicitud, "respuesta": str(respuesta)})
        self.turnos = self.turnos[-self.ventana:]

    def como_texto(self) -> str:
        if not self.turnos:
            return "(sin turnos previos en esta sesión)"
        return "\n".join(f"[{t['hora']}] Técnico: {t['solicitud']}\n"
                         f"[{t['hora']}] Sistema: {t['respuesta'][:self.max_chars]}"
                         for t in self.turnos)

    def __len__(self):
        return len(self.turnos)

## 4. Herramientas

Cada herramienta es una función de Python envuelta como `BaseTool` de CrewAI. Cada especialista recibe solo las que necesita.

| Herramienta | Tipo | Agente | Qué hace |
|---|---|---|---|
| `buscar_manual` | Consulta | Técnico Diagnosticador | Busca en el manual y devuelve fragmentos con página |
| `buscar_historial_fallas` | Consulta | Técnico Diagnosticador | Busca fallas parecidas atendidas antes |
| `consultar_stock` | Consulta | Planificador de Mantenimiento | Stock, ubicación, plazo y alternativa del repuesto |
| `evaluar_criticidad` | Razonamiento | Planificador de Mantenimiento | Clasifica la falla en ALTA, MEDIA o BAJA |
| `crear_orden_trabajo` | Escritura | Coordinador de Mantenimiento | Crea la OT y la guarda en la memoria |
| `enviar_correo` | Escritura | Coordinador de Mantenimiento | Avisa a supervisión o a bodega (simulado) |

Cada vez que se usa una herramienta queda anotado qué agente la usó y a qué hora (`TRAZA_ACTUAL`).

Algunas reglas están en el código y no dependen del modelo:
- La criticidad final es la más alta entre la que propone el agente y la que dan las reglas. Se puede subir, pero no bajar. Una falla ALTA siempre queda esperando aprobación.
- Solo un reporte de falla puede crear una OT.
- Si ya hay una OT abierta para la misma aeronave y el mismo componente, no se crea otra.

In [ ]:
# Lógica de negocio
NIVELES = {"BAJA": 0, "MEDIA": 1, "ALTA": 2}
ESTADOS_ACTIVOS = {"ABIERTA", "PENDIENTE_APROBACION_SUPERVISOR", "EN_ESPERA_REPUESTO"}

# Solicitud actual: la fija el código, no el modelo
SOLICITUD_ACTUAL = {"texto": ""}
TRAZA_ACTUAL = []

PATRON_REPORTE = re.compile(r"\breport(o|a|amos|an)\b|falla en (la )?aeronave|se encendi|se enciende|"
                            r"no entrega|disparo", re.I)

def es_reporte_de_falla(texto: str) -> bool:
    return bool(PATRON_REPORTE.search(texto or ""))

def estado_desde_texto(texto: str) -> str:
    return "en vuelo" if re.search(r"en vuelo", texto or "", re.I) else "en tierra"

def fn_buscar_historial(consulta: str) -> str:
    hits = memoria_lp.buscar(consulta, k=3)
    if not hits:
        return "No hay fallas similares registradas en el historial."
    return "\n".join(f"- ({d.metadata.get('fecha')}) {d.page_content}" for d, _ in hits)


def fn_consultar_stock(codigo_componente: str) -> str:
    codigo = codigo_componente.strip().upper().replace(" ", "")
    item = ENTORNO["inventario"].get(codigo)
    if item is None:
        return (f"Código '{codigo_componente}' no existe en inventario. "
                f"Códigos válidos: {', '.join(ENTORNO['inventario'])}.")
    return json.dumps({"codigo": codigo, **item,
                       "disponible": item["stock"] > 0}, ensure_ascii=False)


REGLAS_ALTA  = ["cortocircuito", "humo", "quemado", "incendio", "chispa", "sobrecalent",
                "pp2", "barra esencial", "ambos alternadores", "los dos alternadores",
                "pérdida total", "perdida total", "sin 28 v", "batería no carga"]
REGLAS_MEDIA = ["transferencia", "luz", "intermitente", "caída de tensión", "caida de tension",
                "tensión baja", "tension baja", "breaker", "disparo", "alternador",
                "tr1", "tr2", "contactor", "relé", "rele"]

def fn_evaluar_criticidad(descripcion_falla: str, estado_aeronave: str = "en tierra") -> str:
    t = descripcion_falla.lower()
    motivos_alta  = [k for k in REGLAS_ALTA  if k in t]
    motivos_media = [k for k in REGLAS_MEDIA if k in t]
    if motivos_alta:
        nivel, motivos = "ALTA", motivos_alta
    elif motivos_media:
        nivel, motivos = "MEDIA", motivos_media
    else:
        nivel, motivos = "BAJA", ["sin indicadores de riesgo en la descripción"]
    # En vuelo la criticidad sube un nivel
    if "vuelo" in estado_aeronave.lower() and nivel != "ALTA":
        nivel = "ALTA" if nivel == "MEDIA" else "MEDIA"
        motivos.append("falla ocurrida en vuelo (+1 nivel)")
    politica = {
        "ALTA":  "Aeronave NO apta para vuelo (AOG). OT queda pendiente de aprobación de supervisión. Notificar a supervisión de inmediato.",
        "MEDIA": "Abrir OT normal; plazo de atención 48 h.",
        "BAJA":  "Programar en la próxima inspección; no requiere notificación.",
    }[nivel]
    return json.dumps({"criticidad": nivel, "motivos": motivos, "politica": politica},
                      ensure_ascii=False)


def fn_crear_orden_trabajo(aeronave: str, componente: str, descripcion: str,
                           criticidad: str, accion_recomendada: str) -> str:
    solicitud = SOLICITUD_ACTUAL["texto"]
    if solicitud and not es_reporte_de_falla(solicitud):
        return ("RECHAZADO: la solicitud actual no es un reporte de falla; "
                "no corresponde crear una orden de trabajo.")
    ordenes = _leer_json(ARCH_OT)
    codigo = componente.strip().upper().replace(" ", "")
    aeronave = aeronave.strip().upper()
    item = ENTORNO["inventario"].get(codigo)

    # Si ya hay una OT activa igual, no se crea otra
    for o in ordenes:
        if o["aeronave"] == aeronave and o["componente"] == codigo and o["estado"] in ESTADOS_ACTIVOS:
            return json.dumps({**o, "nota": "Ya existía una OT activa para esta aeronave y "
                                            "componente; no se duplicó."}, ensure_ascii=False)

    # Se usa la criticidad más alta entre la del agente y la de las reglas
    propuesta = criticidad.strip().upper()
    if propuesta not in NIVELES:
        propuesta = "BAJA"
    base = solicitud or descripcion
    reglas = json.loads(fn_evaluar_criticidad(base, estado_desde_texto(base)))["criticidad"]
    final = max(propuesta, reglas, key=NIVELES.get)

    # El estado lo decide el código
    if final == "ALTA":
        estado = "PENDIENTE_APROBACION_SUPERVISOR"
    elif item is not None and item["stock"] == 0:
        estado = "EN_ESPERA_REPUESTO"
    else:
        estado = "ABIERTA"
    hoy = datetime.now()
    ot = {
        "id": f"OT-{hoy:%Y%m%d}-{len(ordenes) + 1:03d}",
        "fecha": hoy.strftime("%Y-%m-%d %H:%M"),
        "aeronave": aeronave, "componente": codigo, "descripcion": descripcion,
        "criticidad": final, "criticidad_propuesta_agente": propuesta, "criticidad_reglas": reglas,
        "ajustada_por_compuerta": final != propuesta,
        "accion_recomendada": accion_recomendada, "estado": estado,
    }
    ordenes.append(ot)
    _guardar_json(ARCH_OT, ordenes)
    memoria_lp.recordar(f"{aeronave} | {descripcion}. Componente {codigo}. Acción: "
                        f"{accion_recomendada}. Criticidad {final}. {ot['id']} ({estado}).",
                        tipo="orden_trabajo", ot=ot["id"])
    return json.dumps(ot, ensure_ascii=False)


def fn_enviar_correo(destinatario: str, asunto: str, cuerpo: str) -> str:
    correos = _leer_json(ARCH_CORREOS)
    ts = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    correos.append({"timestamp": ts, "para": destinatario, "asunto": asunto, "cuerpo": cuerpo})
    _guardar_json(ARCH_CORREOS, correos)
    print(f"\n[Correo simulado] Para: {destinatario} | Asunto: {asunto}\n{cuerpo[:400]}\n")
    return f"Correo enviado a {destinatario} con asunto '{asunto}' ({ts})."

In [ ]:
# Herramientas para CrewAI
# El modelo elige cada herramienta según su descripción

class _ConsultaInput(BaseModel):
    consulta: str = Field(description="Texto a buscar.")
class _StockInput(BaseModel):
    codigo_componente: str = Field(description="Código del repuesto, ej: TR2, 49P, 33P5.")
class _CriticidadInput(BaseModel):
    descripcion_falla: str = Field(description="Descripción de la falla.")
    estado_aeronave: str = Field(description="'en tierra' o 'en vuelo'.")
class _OTInput(BaseModel):
    aeronave: str = Field(description="Identificador de la aeronave, ej: SH-32-02.")
    componente: str = Field(description="Código de inventario del componente, ej: TR2.")
    descripcion: str = Field(description="Descripción breve de la falla.")
    criticidad: str = Field(description="ALTA, MEDIA o BAJA (de evaluar_criticidad).")
    accion_recomendada: str = Field(description="Acción de mantenimiento recomendada.")
class _CorreoInput(BaseModel):
    destinatario: str = Field(description="Correo del destinatario.")
    asunto: str = Field(description="Asunto.")
    cuerpo: str = Field(description="Cuerpo del mensaje.")


class HerramientaRegistrada(BaseTool):
    # Registra qué agente usó la herramienta y cuándo
    agente: str = ""
    def _registrar(self, **args):
        TRAZA_ACTUAL.append({"herramienta": self.name, "agente": self.agente,
                             "hora": datetime.now().strftime("%H:%M:%S"),
                             "args": {k: str(v)[:120] for k, v in args.items()}})

class ToolManual(HerramientaRegistrada):
    name: str = "buscar_manual"
    description: str = ("Busca en el Manual Eléctrico SH-32 (RAG). Úsala para cualquier pregunta técnica "
                        "del sistema eléctrico: barras, alternadores, TR, contactores, relés, batería. "
                        "Devuelve fragmentos con número de página, o SIN_RESULTADOS_RELEVANTES.")
    args_schema: Type[BaseModel] = _ConsultaInput
    def _run(self, consulta: str) -> str:
        self._registrar(consulta=consulta)
        return buscar_en_manual(consulta)

class ToolHistorial(HerramientaRegistrada):
    name: str = "buscar_historial_fallas"
    description: str = "Busca fallas similares atendidas antes en la flota (memoria de largo plazo)."
    args_schema: Type[BaseModel] = _ConsultaInput
    def _run(self, consulta: str) -> str:
        self._registrar(consulta=consulta)
        return fn_buscar_historial(consulta)

class ToolStock(HerramientaRegistrada):
    name: str = "consultar_stock"
    description: str = ("Consulta en bodega el stock de un repuesto por su código (ej: 33P5, TR2, 49P, "
                        "BAT-24V): disponibilidad, ubicación, días de reposición y alternativa.")
    args_schema: Type[BaseModel] = _StockInput
    def _run(self, codigo_componente: str) -> str:
        self._registrar(codigo_componente=codigo_componente)
        return fn_consultar_stock(codigo_componente)

class ToolCriticidad(HerramientaRegistrada):
    name: str = "evaluar_criticidad"
    description: str = ("Clasifica la criticidad de una falla (ALTA/MEDIA/BAJA) con las reglas del taller. "
                        "Si la falla ocurrió en vuelo, la criticidad sube un nivel.")
    args_schema: Type[BaseModel] = _CriticidadInput
    def _run(self, descripcion_falla: str, estado_aeronave: str = "en tierra") -> str:
        self._registrar(descripcion_falla=descripcion_falla, estado_aeronave=estado_aeronave)
        return fn_evaluar_criticidad(descripcion_falla, estado_aeronave)

class ToolOT(HerramientaRegistrada):
    name: str = "crear_orden_trabajo"
    description: str = ("Crea la orden de trabajo (OT) en el sistema del taller. El estado lo fija el "
                        "sistema según criticidad y stock. Devuelve la OT y su estado.")
    args_schema: Type[BaseModel] = _OTInput
    def _run(self, aeronave: str, componente: str, descripcion: str, criticidad: str,
             accion_recomendada: str) -> str:
        self._registrar(aeronave=aeronave, componente=componente, criticidad=criticidad)
        return fn_crear_orden_trabajo(aeronave, componente, descripcion, criticidad, accion_recomendada)

class ToolCorreo(HerramientaRegistrada):
    name: str = "enviar_correo"
    description: str = "Envía un correo (simulado) a supervisión o abastecimiento."
    args_schema: Type[BaseModel] = _CorreoInput
    def _run(self, destinatario: str, asunto: str, cuerpo: str) -> str:
        self._registrar(destinatario=destinatario, asunto=asunto)
        return fn_enviar_correo(destinatario, asunto, cuerpo)

print("Herramientas CrewAI:", [t.name for t in (ToolManual(), ToolHistorial(), ToolStock(),
                                                ToolCriticidad(), ToolOT(), ToolCorreo())])

Herramientas CrewAI: ['buscar_manual', 'buscar_historial_fallas', 'consultar_stock', 'evaluar_criticidad', 'crear_orden_trabajo', 'enviar_correo']


### 4.1 Pruebas de las herramientas

Estas pruebas no usan el modelo. Se hacen sobre un archivo temporal para no ensuciar las órdenes ni la memoria, y el resultado se guarda en `evidencias/pruebas_herramientas.csv`.

In [ ]:
class _MemoriaNula:   # las pruebas no guardan nada en la memoria
    def recordar(self, *a, **k): pass

_respaldo = (ARCH_OT, memoria_lp, ENTORNO["inventario"]["TR2"]["stock"])
ARCH_OT, memoria_lp = Path(tempfile.mkdtemp()) / "ot_prueba.json", _MemoriaNula()
resultados_pruebas = []

def prueba(nombre, condicion):
    resultados_pruebas.append({"prueba": nombre, "resultado": "OK" if condicion else "FALLA"})

try:
    prueba("Stock de TR2 disponible", json.loads(fn_consultar_stock("tr2"))["disponible"] is True)
    prueba("Código inexistente informa los válidos", "no existe" in fn_consultar_stock("XYZ"))
    prueba("Luz de transferencia en tierra = MEDIA",
           json.loads(fn_evaluar_criticidad("luz de transferencia intermitente"))["criticidad"] == "MEDIA")
    prueba("La misma falla en vuelo sube a ALTA",
           json.loads(fn_evaluar_criticidad("luz de transferencia intermitente", "en vuelo"))["criticidad"] == "ALTA")

    SOLICITUD_ACTUAL["texto"] = "Reporto falla en SH-32-02: la luz de transferencia se encendió EN VUELO."
    ot1 = json.loads(fn_crear_orden_trabajo("SH-32-02", "49P", "luz de transferencia", "MEDIA", "revisar 49P"))
    prueba("El agente propone MEDIA en vuelo y queda ALTA",
           ot1["criticidad"] == "ALTA" and ot1["ajustada_por_compuerta"])
    prueba("OT ALTA queda pendiente de aprobación", ot1["estado"] == "PENDIENTE_APROBACION_SUPERVISOR")
    ot2 = json.loads(fn_crear_orden_trabajo("SH-32-02", "49P", "luz de transferencia", "ALTA", "revisar 49P"))
    prueba("Idempotencia: no duplica la OT", ot2["id"] == ot1["id"] and "nota" in ot2)

    ENTORNO["inventario"]["TR2"]["stock"] = 0
    SOLICITUD_ACTUAL["texto"] = "Reporto falla en SH-32-03, en tierra: TR2 no entrega 28 V."
    ot3 = json.loads(fn_crear_orden_trabajo("SH-32-03", "TR2", "TR2 sin salida", "MEDIA", "reemplazar TR2"))
    prueba("Sin stock, la OT queda en espera", ot3["estado"] == "EN_ESPERA_REPUESTO")

    SOLICITUD_ACTUAL["texto"] = "¿Cuántos perros tienen el pelaje blanco?"
    prueba("Una pregunta (no reporte) no puede crear OT",
           fn_crear_orden_trabajo("SH-32-01", "49P", "x", "BAJA", "x").startswith("RECHAZADO"))
finally:
    ARCH_OT, memoria_lp = _respaldo[0], _respaldo[1]
    ENTORNO["inventario"]["TR2"]["stock"] = _respaldo[2]
    SOLICITUD_ACTUAL["texto"] = ""

df_pruebas = pd.DataFrame(resultados_pruebas)
display(df_pruebas)
df_pruebas.to_csv(DIR_EVIDENCIA / "pruebas_herramientas.csv", index=False)
print(f"{(df_pruebas.resultado == 'OK').sum()}/{len(df_pruebas)} pruebas correctas")

,prueba,resultado
0,Stock de TR2 disponible,OK
1,Código inexistente informa los válidos,OK
2,Luz de transferencia en tierra = MEDIA,OK
3,La misma falla en vuelo sube a ALTA,OK
4,El agente propone MEDIA en vuelo y queda ALTA,OK
5,OT ALTA queda pendiente de aprobación,OK
6,Idempotencia: no duplica la OT,OK
7,"Sin stock, la OT queda en espera",OK
8,Una pregunta (no reporte) no puede crear OT,OK


9/9 pruebas correctas


## 5. Equipo de agentes

| Agente | Qué hace | Herramientas |
|---|---|---|
| Jefe de Taller (Manager) | Decide qué hacer con cada solicitud, reparte el trabajo y revisa los resultados | Ninguna, solo delega |
| Técnico Diagnosticador | Encuentra el sistema y el componente con el manual y el historial | `buscar_manual`, `buscar_historial_fallas` |
| Planificador de Mantenimiento | Decide la criticidad y revisa si hay repuesto | `evaluar_criticidad`, `consultar_stock` |
| Coordinador de Mantenimiento | Crea la orden de trabajo y avisa | `crear_orden_trabajo`, `enviar_correo` |

No hay un plan fijo en el código. El Manager recibe una sola tarea con las reglas del taller, la conversación reciente y los casos parecidos, y decide los pasos. Un reporte de falla pasa por los tres especialistas, una pregunta sin relación solo por el Diagnosticador y un seguimiento se responde con la memoria. Si aparece algo nuevo, como una falla en vuelo o un repuesto sin stock, cambia lo que hace después.

Se usa un Manager propio (`manager_agent`) para darle el rol de Jefe de Taller y limitarlo a 10 iteraciones, porque cada delegación es otra llamada al modelo.

Al terminar cada solicitud, el código revisa los correos: si quedó una OT grave sin aviso a supervisión, o una OT sin stock sin aviso a bodega, lo envía y lo anota.

In [ ]:
DESCRIPCION_TAREA = """Fecha: {fecha}. Atiende esta solicitud del taller de mantenimiento eléctrico de la flota SH-32.

SOLICITUD: {solicitud}

Conversación reciente de esta sesión (memoria de corto plazo):
{historial}

Casos similares recuperados de la memoria de largo plazo:
{contexto_lp}

Planifica los pasos, decide a qué especialista delegar cada uno y en qué orden, y revisa sus resultados.
Políticas obligatorias:
1. Reporte de falla: (a) el Técnico Diagnosticador identifica el sistema y el código del componente con
   el manual (citando páginas) y el historial; (b) el Planificador de Mantenimiento evalúa la criticidad
   (indicando si ocurrió en vuelo o en tierra) y consulta el stock de ese código; (c) el Coordinador de
   Mantenimiento crea la OT con esa criticidad y envía los correos que correspondan.
2. Criticidad ALTA: el Coordinador envía correo a {correo_supervision} pidiendo aprobación. Nadie del
   equipo aprueba OT críticas.
3. Repuesto sin stock: el Coordinador envía correo a {correo_bodega} con el plazo de reposición e
   informa la alternativa aprobada si existe.
4. Pregunta técnica sin falla: solo el Técnico Diagnosticador con el manual; no se crea OT.
5. Tema ajeno al sistema eléctrico SH-32 (o el manual responde SIN_RESULTADOS_RELEVANTES): declina
   amablemente, sin delegar tareas de escritura.
6. Seguimiento de algo ya conversado: responde con la conversación reciente y la memoria de largo
   plazo; no crees OT nuevas.
7. Nunca inventes códigos de componente, valores eléctricos ni números de OT.
Al delegar, entrega al especialista TODO el contexto que necesita (aeronave, si fue en vuelo o en tierra,
código del componente, criticidad): los especialistas no ven esta conversación."""

RESULTADO_ESPERADO = ("Respuesta en español con cuatro apartados: **Diagnóstico**, **Decisiones tomadas** "
                      "(qué se delegó a quién y por qué, incluido cualquier cambio de plan), "
                      "**Próximos pasos** y **Fuentes** (páginas del manual y números de OT).")

DELEGACIONES = []

if crewai_event_bus is not None and not globals().get("_ESCUCHA_DELEGACIONES"):
    @crewai_event_bus.on(ToolUsageStartedEvent)
    def _registrar_delegacion(source, event):
        # Guarda cada delegación del Manager
        try:
            if "coworker" in (event.tool_name or "").lower():
                args = event.tool_args
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except Exception:
                        args = {"task": args}
                args = args if isinstance(args, dict) else {}
                DELEGACIONES.append({"hora": datetime.now().strftime("%H:%M:%S"), "accion": event.tool_name,
                                     "a": str(args.get("coworker", "")),
                                     "tarea": str(args.get("task", args.get("question", "")))[:200]})
        except Exception:
            pass
    _ESCUCHA_DELEGACIONES = True


def construir_equipo() -> Crew:
    """Crea el equipo de nuevo para cada solicitud."""
    llm_w = crear_llm_crewai(max_tokens=600)
    diagnosticador = Agent(
        role="Técnico Diagnosticador",
        goal="Identificar el sistema y el componente probable de la falla, solo con evidencia del manual y del historial.",
        backstory="Técnico eléctrico con años en la flota SH-32. Nunca inventa códigos: cita la página del manual.",
        tools=[ToolManual(agente="Técnico Diagnosticador"), ToolHistorial(agente="Técnico Diagnosticador")],
        llm=llm_w, max_iter=4, allow_delegation=False, verbose=False)
    planificador_mant = Agent(
        role="Planificador de Mantenimiento",
        goal="Determinar la criticidad de la falla y la disponibilidad del repuesto, y recomendar la acción.",
        backstory="Responsable de priorizar trabajos según seguridad y disponibilidad de bodega.",
        tools=[ToolCriticidad(agente="Planificador de Mantenimiento"),
               ToolStock(agente="Planificador de Mantenimiento")],
        llm=llm_w, max_iter=4, allow_delegation=False, verbose=False)
    coordinador = Agent(
        role="Coordinador de Mantenimiento",
        goal="Registrar la orden de trabajo y notificar a quien corresponda según la política.",
        backstory="Coordina el taller: si la criticidad es ALTA avisa a supervisión; si no hay stock avisa a abastecimiento.",
        tools=[ToolOT(agente="Coordinador de Mantenimiento"), ToolCorreo(agente="Coordinador de Mantenimiento")],
        llm=crear_llm_crewai(max_tokens=900), max_iter=4, allow_delegation=False, verbose=False)
    manager = Agent(
        role="Jefe de Taller (Manager)",
        goal=("Planificar la atención de cada solicitud, delegar cada paso al especialista adecuado en el "
              "orden correcto y entregar una respuesta final verificada."),
        backstory=("Jefe del taller de mantenimiento eléctrico SH-32. No usa herramientas: coordina al equipo, "
                   "prioriza la seguridad y respeta que las OT críticas las aprueba una persona."),
        llm=crear_llm_crewai(max_tokens=800, temperature=0), allow_delegation=True, max_iter=10, verbose=False)
    tarea = Task(description=DESCRIPCION_TAREA, expected_output=RESULTADO_ESPERADO)   # sin agente: la reparte el Manager
    # Sin registros detallados: al final solo se muestra el resumen de cada solicitud
    return Crew(agents=[diagnosticador, planificador_mant, coordinador], tasks=[tarea],
                process=Process.hierarchical, manager_agent=manager,
                max_rpm=10, verbose=False, tracing=False)


class SistemaJerarquico:
    def __init__(self, nombre_sesion: str = "sesion-1"):
        self.sesion = nombre_sesion
        self.memoria_cp = MemoriaCortoPlazo(ventana=6)

    def _contexto_largo_plazo(self, solicitud: str) -> str:
        hits = memoria_lp.buscar(solicitud, k=2)
        return "\n".join(f"- ({d.metadata.get('fecha')}) {d.page_content}" for d, _ in hits) or "(sin episodios similares)"

    def _auditar(self, ots: list, correos: list) -> list:
        """Envía los correos obligatorios que el equipo no envió."""
        correcciones, destinos = [], " ".join(c["para"] for c in correos)
        for ot in ots:
            if ot["estado"] == "PENDIENTE_APROBACION_SUPERVISOR" and CORREO_SUPERVISION not in destinos:
                fn_enviar_correo(CORREO_SUPERVISION, f"[Auditoría] Aprobación requerida {ot['id']}",
                                 f"La OT {ot['id']} ({ot['aeronave']}, {ot['componente']}) quedó con criticidad "
                                 "ALTA y requiere aprobación. Enviado por la auditoría de políticas.")
                correcciones.append(f"correo a supervisión por {ot['id']}")
                destinos += " " + CORREO_SUPERVISION
            if ot["estado"] == "EN_ESPERA_REPUESTO" and CORREO_BODEGA not in destinos:
                fn_enviar_correo(CORREO_BODEGA, f"[Auditoría] Repuesto sin stock {ot['componente']}",
                                 f"La OT {ot['id']} espera el repuesto {ot['componente']}. "
                                 "Enviado por la auditoría de políticas.")
                correcciones.append(f"correo a abastecimiento por {ot['id']}")
                destinos += " " + CORREO_BODEGA
        return correcciones

    def atender(self, solicitud: str, mostrar: bool = True) -> dict:
        t0 = time.time()
        TRAZA_ACTUAL.clear(); DELEGACIONES.clear()
        SOLICITUD_ACTUAL["texto"] = solicitud
        n_ot, n_correos = len(_leer_json(ARCH_OT)), len(_leer_json(ARCH_CORREOS))
        entradas = {"fecha": datetime.now().strftime("%d/%m/%Y"), "solicitud": solicitud,
                    "historial": self.memoria_cp.como_texto(),
                    "contexto_lp": self._contexto_largo_plazo(solicitud),
                    "correo_supervision": CORREO_SUPERVISION, "correo_bodega": CORREO_BODEGA}
        respuesta, uso, error = "", None, None
        for intento in (1, 2):
            try:
                salida = en_hilo(construir_equipo().kickoff, inputs=entradas)
                respuesta = limpiar(getattr(salida, "raw", salida))
                uso, error = getattr(salida, "token_usage", None), None
                break
            except Exception as e:
                error = f"{type(e).__name__}: {str(e)[:300]}"
                txt = str(e).lower()
                cuota_diaria = "per day" in txt or "tpd" in txt
                if intento == 1 and ("429" in txt or "rate" in txt) and not cuota_diaria:
                    print("Límite por minuto del proveedor, se reintenta en 60 segundos.")
                    time.sleep(60)
                    continue
                break
        if error:
            print("El equipo no terminó la solicitud:", error)
            if "per day" in error.lower() or "tpd" in error.lower():
                print("   Se agotó la cuota DIARIA del proveedor: espera a que se renueve o usa una key de Mistral.")
            respuesta = respuesta or f"(sin respuesta: {error})"

        ots = _leer_json(ARCH_OT)[n_ot:]
        correcciones = self._auditar(ots, _leer_json(ARCH_CORREOS)[n_correos:])
        correos = _leer_json(ARCH_CORREOS)[n_correos:]
        if not error:   # las solicitudes fallidas no se guardan
            resumen_ot = "; ".join(f"{o['id']} {o['componente']} {o['estado']}" for o in ots)
            self.memoria_cp.guardar(solicitud, (f"[OT: {resumen_ot}] " if resumen_ot else "") + respuesta)
        SOLICITUD_ACTUAL["texto"] = ""

        agentes = list(dict.fromkeys(u["agente"] for u in TRAZA_ACTUAL))
        traza = {"sesion": self.sesion, "fecha": datetime.now().isoformat(timespec="seconds"),
                 "solicitud": solicitud, "es_reporte_de_falla": es_reporte_de_falla(solicitud),
                 "herramientas": list(TRAZA_ACTUAL), "secuencia": [u["herramienta"] for u in TRAZA_ACTUAL],
                 "agentes": agentes, "delegaciones": list(DELEGACIONES),
                 "ots_creadas": [{k: o[k] for k in ("id", "aeronave", "componente", "criticidad", "estado",
                                                     "ajustada_por_compuerta")} for o in ots],
                 "correos": [{"para": c["para"], "asunto": c["asunto"]} for c in correos],
                 "correcciones_auditoria": correcciones,
                 "tokens": getattr(uso, "total_tokens", None),
                 "llamadas_LLM": getattr(uso, "successful_requests", None),
                 "segundos": round(time.time() - t0, 1), "error": error, "respuesta": respuesta}
        with ARCH_TRAZAS.open("a", encoding="utf-8") as f:
            f.write(json.dumps(traza, ensure_ascii=False) + "\n")
        if mostrar:
            self.mostrar(traza)
        return traza

    @staticmethod
    def mostrar(traza: dict):
        filas = "\n".join(f"| {i} | {u['agente']} | `{u['herramienta']}` | {u['hora']} |"
                          for i, u in enumerate(traza["herramientas"], 1)) or "| - | - | (sin herramientas) | - |"
        ots = ", ".join(f"{o['id']} → **{o['estado']}** ({o['criticidad']}"
                        + (", ajustada por la compuerta" if o["ajustada_por_compuerta"] else "") + ")"
                        for o in traza["ots_creadas"]) or "ninguna"
        correos = ", ".join(c["para"] for c in traza["correos"]) or "ninguno"
        corr = ", ".join(traza["correcciones_auditoria"]) or "ninguna"
        display(Markdown(
            f"### Lo que decidió el Manager\n"
            f"| # | Agente | Herramienta | Hora |\n|---|---|---|---|\n{filas}\n\n"
            f"**Delegaciones del Manager:** {' → '.join(d['a'] for d in traza['delegaciones']) or 'ninguna'}  \n"
            f"**Agentes que usaron herramientas:** {' → '.join(traza['agentes']) or 'ninguno (respondió el Manager)'}  \n"
            f"**OT creadas:** {ots}  \n**Correos:** {correos}  \n**Correcciones de la auditoría:** {corr}  \n"
            f"**Tokens:** {traza['tokens']} · **Llamadas al modelo:** {traza['llamadas_LLM']} · "
            f"**Tiempo:** {traza['segundos']} s\n\n---\n### Respuesta del equipo\n{traza['respuesta']}"))


sistema = SistemaJerarquico("sesion-1")
print("Sistema listo: Manager, Técnico Diagnosticador, Planificador de Mantenimiento "
      "y Coordinador de Mantenimiento.")

Sistema listo: Manager, Técnico Diagnosticador, Planificador de Mantenimiento y Coordinador de Mantenimiento.


## 6. Escenarios de prueba

Cada escenario cambia una sola condición para ver si el agente toma otro camino.

| Escenario | Situación | Resultado esperado |
|---|---|---|
| A | Falla en tierra, con stock | Pasa por los tres especialistas y la OT queda abierta, sin correos |
| B | La misma falla, pero en vuelo | Sube a ALTA, la OT espera aprobación y se avisa a supervisión |
| C | Repuesto sin stock | La OT queda en espera y se avisa a bodega |
| D | Pregunta sin relación con el manual | Declina y no crea nada |
| E | Pregunta de seguimiento | Responde con la memoria de la sesión, sin crear OT |

`REINICIAR_EVIDENCIAS = True` borra las órdenes, correos y registros anteriores para empezar limpio. La memoria de largo plazo se mantiene.

In [ ]:
REINICIAR_EVIDENCIAS = True
if REINICIAR_EVIDENCIAS:
    for arch in (ARCH_OT, ARCH_CORREOS, ARCH_TRAZAS):
        arch.unlink(missing_ok=True)

# Escenario A: falla en tierra
traza_A = sistema.atender(
    "Reporto falla en la aeronave SH-32-01, en tierra: durante la prueba en plataforma "
    "se enciende la luz de transferencia de la red 1. Sospechamos del relé 49P.")

Maximum iterations reached. Requesting final answer.
Maximum iterations reached. Requesting final answer.


### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| 1 | Técnico Diagnosticador | `buscar_manual` | 03:57:19 |
| 2 | Técnico Diagnosticador | `buscar_manual` | 03:57:36 |
| 3 | Técnico Diagnosticador | `buscar_historial_fallas` | 03:58:09 |
| 4 | Técnico Diagnosticador | `buscar_manual` | 03:58:27 |
| 5 | Técnico Diagnosticador | `buscar_manual` | 03:59:36 |
| 6 | Técnico Diagnosticador | `buscar_manual` | 03:59:43 |
| 7 | Técnico Diagnosticador | `buscar_manual` | 04:00:18 |
| 8 | Técnico Diagnosticador | `buscar_historial_fallas` | 04:00:55 |
| 9 | Planificador de Mantenimiento | `evaluar_criticidad` | 04:02:26 |
| 10 | Planificador de Mantenimiento | `consultar_stock` | 04:02:27 |
| 11 | Coordinador de Mantenimiento | `crear_orden_trabajo` | 04:03:27 |

**Delegaciones del Manager:** Técnico Diagnosticador → Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**Agentes que usaron herramientas:** Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**OT creadas:** OT-20261023-001 → **ABIERTA** (MEDIA)  
**Correos:** ninguno  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 17463 · **Llamadas al modelo:** 5 · **Tiempo:** 394.2 s

---
### Respuesta del equipo
**Diagnóstico**
La falla reportada en la aeronave SH-32-01 corresponde a la iluminación de la luz de transferencia de la red 1 durante una prueba en plataforma (en tierra). El componente identificado es el relé de transferencia 49P, perteneciente al sistema de transferencia automática de la red eléctrica. Existe una coincidencia directa con el historial de la misma aeronave (OT-20260314-001, 14/03/2026), donde se reemplazó este mismo relé por una falla similar detectada en vuelo. El diagnóstico técnico confirma la sospecha inicial del equipo de taller.

**Decisiones tomadas**
1.  **Delegación al Técnico Diagnosticador:** Se le solicitó la identificación del sistema y código del componente. Confirmó el código 49P y el sistema de transferencia automática. No fue posible localizar la página específica del manual para el 49P (solo se encontraron referencias a relés 53X1/53X2 en páginas 13 y 6), pero la coincidencia con el historial validó el diagnóstico.
2.  **Delegación al Planificador de Mantenimiento:** Se le solicitó evaluar la criticidad y el stock. Asignó criticidad **MEDIA** porque la falla se detectó en tierra (no aplica el incremento de nivel por falla en vuelo) y confirmó que hay **3 unidades disponibles** del código 49P en la Bodega A-07.
3.  **Delegación al Coordinador de Mantenimiento:** Se le solicitó crear la OT y gestionar la comunicación. Creó la **OT-20261023-001** con criticidad MEDIA. No se enviaron correos a supervisión (no es criticidad ALTA) ni a abastecimiento (hay stock disponible). La OT quedó registrada con estado ABIERTA.

**Próximos pasos**
*   Proceder con la inspección y posible reemplazo del relé 49P en la aeronave SH-32-01 dentro del plazo de 48 horas establecido para criticidad MEDIA.
*   Verificar el funcionamiento correcto de la transferencia de la red 1 tras la intervención.
*   Cerrar la OT-20261023-001 una vez confirmada la resolución de la falla.

**Fuentes**
*   **Manual Técnico SH-32:** Páginas 13 y 6 (referencias al sistema de transferencia automática; no se localizó página específica para el relé 49P).
*   **Historial de Mantenimiento:** OT-20260314-001 (14/03/2026, SH-32-01, reemplazo de relé 49P).
*   **Orden de Trabajo Actual:** OT-20261023-001 (creada el 23/10/2026).

In [ ]:
pausa()
# Escenario B: la misma falla, pero en vuelo
traza_B = sistema.atender(
    "Reporto falla en la aeronave SH-32-02: la luz de transferencia de la red 1 se encendió "
    "EN VUELO y la tripulación informó pérdida de tensión. Sospechamos del relé 49P.")

Pausa de 30 segundos (límite de tokens por minuto).
Maximum iterations reached. Requesting final answer.

[Correo simulado] Para: supervision.mantenimiento@taller-sh32.cl | Asunto: Solicitud de aprobación OT-20261023-002 - SH-32-02 - Criticidad ALTA
Estimados,

Se solicita la aprobación de la Orden de Trabajo OT-20261023-002 para la aeronave SH-32-02.

Detalles:
- Falla: Luz de transferencia de la red 1 encendida EN VUELO con pérdida de tensión.
- Componente: Relé de transferencia 49P.
- Criticidad: ALTA.
- Acción recomendada: Reemplazo del Relé de transferencia 49P y verificación de la red eléctrica 1.
- Stock: Disponible (3 unidades en Bode



### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| 1 | Técnico Diagnosticador | `buscar_manual` | 04:04:53 |
| 2 | Técnico Diagnosticador | `buscar_historial_fallas` | 04:05:01 |
| 3 | Técnico Diagnosticador | `buscar_historial_fallas` | 04:05:11 |
| 4 | Técnico Diagnosticador | `buscar_manual` | 04:05:30 |
| 5 | Planificador de Mantenimiento | `evaluar_criticidad` | 04:07:25 |
| 6 | Planificador de Mantenimiento | `consultar_stock` | 04:07:34 |
| 7 | Coordinador de Mantenimiento | `crear_orden_trabajo` | 04:08:49 |
| 8 | Coordinador de Mantenimiento | `enviar_correo` | 04:08:51 |

**Delegaciones del Manager:** Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**Agentes que usaron herramientas:** Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**OT creadas:** OT-20261023-002 → **PENDIENTE_APROBACION_SUPERVISOR** (ALTA)  
**Correos:** supervision.mantenimiento@taller-sh32.cl  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 15063 · **Llamadas al modelo:** 4 · **Tiempo:** 322.4 s

---
### Respuesta del equipo
**Diagnóstico**
La falla reportada en la aeronave SH-32-02 corresponde a la iluminación de la luz de transferencia de la red 1 y la pérdida de tensión, ocurrida **en vuelo**. El componente identificado es el **relé de transferencia 49P**, perteneciente al sistema de transferencia automática entre redes (200 V / 400 Hz). El diagnóstico técnico confirma la sospecha inicial del equipo de taller, basándose en el manual de mantenimiento (páginas 3, 8 y 12) que describe cómo la pérdida de tensión en una barra activa el relé 49P para transferir la carga y encender la luz de advertencia. No se encontraron antecedentes previos de este componente específico en la SH-32-02, aunque la aeronave tiene un historial de fallas eléctricas críticas (TR2 en mayo de 2026).

**Decisiones tomadas**
1.  **Delegación al Técnico Diagnosticador:** Se le solicitó confirmar el sistema y el código del componente (49P) citando el manual y revisando el historial. Esto fue necesario para validar la sospecha inicial antes de planificar la reparación.
2.  **Delegación al Planificador de Mantenimiento:** Se le pidió evaluar la criticidad y el stock. Determinó que la criticidad es **ALTA** debido a que la falla ocurrió en vuelo y afectó la tensión de la red. Confirmó que hay **3 unidades disponibles** del repuesto 49P en la Bodega A-07, por lo que no se requiere pedido a abastecimiento.
3.  **Delegación al Coordinador de Mantenimiento:** Se le instruyó crear la OT con criticidad ALTA y enviar el correo de aprobación a supervisión. Según la política, las OT críticas no pueden ser aprobadas por el equipo de taller, por lo que se envió la solicitud a `supervision.mantenimiento@taller-sh32.cl`. La OT **OT-20261023-002** quedó en estado "PENDIENTE_APROBACION_SUPERVISOR".

**Próximos pasos**
1.  Esperar la aprobación de la OT-20261023-002 por parte de la supervisión de mantenimiento.
2.  Una vez aprobada, retirar el repuesto 49P de la Bodega A-07.
3.  Proceder con el reemplazo del relé de transferencia 49P en la aeronave SH-32-02.
4.  Realizar pruebas de funcionamiento del sistema de transferencia automática para verificar la corrección de la falla.
5.  Registrar la reparación en el historial de la aeronave y devolverla a servicio.

**Fuentes**
*   Manual de Mantenimiento Eléctrico SH-32: Páginas 3 (Sistema de Generación), 8 (Esquema de Distribución y Componente 49P), 12 (Funcionamiento de la Transferencia).
*   Orden de Trabajo: OT-20261023-002 (SH-32-02, Criticidad ALTA, Pendiente de aprobación).
*   Antecedentes: OT-20260502-004 (SH-32-02, falla TR2), OT-20260314-001 (SH-32-01, falla 49P en vuelo), OT-20261023-001

In [ ]:
pausa()
# Escenario C: el TR2 queda sin stock
ENTORNO["inventario"]["TR2"]["stock"] = 0
print("Stock de TR2 cambiado a 0\n")

traza_C = sistema.atender(
    "Reporto falla en la aeronave SH-32-03, en tierra: el transformador-rectificador TR2 "
    "no entrega 28 V en la revisión de hangar, la barra queda con tensión baja.")

Pausa de 30 segundos (límite de tokens por minuto).
Stock de TR2 cambiado a 0

Maximum iterations reached. Requesting final answer.

[Correo simulado] Para: supervision.mantenimiento@taller-sh32.cl | Asunto: APROBACIÓN OT CRÍTICA - SH-32-03 - TR2 - AOG
Estimados,

Se solicita aprobación de la Orden de Trabajo OT-20261023-003 para la aeronave SH-32-03.

Detalles:
- Componente: Transformador-Rectificador TR2
- Falla: TR2 no entrega 28 V, barra con tensión baja (Red trifásica 2 / Barra esencial PP2 de 28 V).
- Criticidad: ALTA
- Estado de aeronave: NO apta para vuelo (AOG).
- Antecedente: SH-32-02 tuvo la misma falla el 02/05/2026 (OT-20260502-004)


[Correo simulado] Para: abastecimiento@taller-sh32.cl | Asunto: FALTA DE STOCK - Repuesto TR2 - SH-32-03 - Plazo 15 días - Sin alternativa técnica
Estimados,

Se informa que no existe stock disponible (0 unidades) del repuesto Transformador-Rectificador TR2, requerido para la aeronave SH-32-03 (OT-20261023-003).

Detalles:
- Componente: TR2
-

### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| 1 | Técnico Diagnosticador | `buscar_manual` | 04:10:43 |
| 2 | Técnico Diagnosticador | `buscar_manual` | 04:10:49 |
| 3 | Técnico Diagnosticador | `buscar_manual` | 04:11:24 |
| 4 | Técnico Diagnosticador | `buscar_historial_fallas` | 04:12:00 |
| 5 | Planificador de Mantenimiento | `evaluar_criticidad` | 04:13:56 |
| 6 | Planificador de Mantenimiento | `consultar_stock` | 04:14:05 |
| 7 | Coordinador de Mantenimiento | `crear_orden_trabajo` | 04:15:33 |
| 8 | Coordinador de Mantenimiento | `enviar_correo` | 04:15:49 |
| 9 | Coordinador de Mantenimiento | `enviar_correo` | 04:16:08 |

**Delegaciones del Manager:** Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**Agentes que usaron herramientas:** Técnico Diagnosticador → Planificador de Mantenimiento → Coordinador de Mantenimiento  
**OT creadas:** OT-20261023-003 → **PENDIENTE_APROBACION_SUPERVISOR** (ALTA)  
**Correos:** supervision.mantenimiento@taller-sh32.cl, abastecimiento@taller-sh32.cl  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 16066 · **Llamadas al modelo:** 4 · **Tiempo:** 385.0 s

---
### Respuesta del equipo
**Diagnóstico**
La falla reportada en la aeronave SH-32-03 corresponde a la ausencia de tensión de 28 V en la salida del transformador-rectificador TR2, lo que deja la barra esencial PP2 con tensión baja. Según el manual de mantenimiento eléctrico SH-32 (páginas 4, 6 y 19), el TR2 forma parte de la red trifásica 2 y es el responsable de alimentar la barra esencial PP2 de 28 V, la cual está acoplada a la batería y alimenta circuitos vitales. La falla coincide con el escenario de "pérdida de un transformador-rectificador" descrito en la página 6, donde los módulos de protección (B1/B2) detectan la caída de tensión y el módulo B4 puede desacoplar el TR2 para aislar un posible cortocircuito. Existe un antecedente directo en la flota: la aeronave hermana SH-32-02 presentó la misma falla el 02/05/2026 (OT-20260502-004), donde se reemplazó el TR2 con criticidad ALTA. No se registran OTs previas para TR2 en la SH-32-03.

**Decisiones tomadas**
1.  **Delegación al Técnico Diagnosticador:** Se le solicitó identificar el sistema y el componente usando el manual e historial. Confirmó que el componente es el TR2 (barra PP2, 28 V) y citó las páginas 4, 6, 19 y 36 del manual. Identificó el antecedente de la SH-32-02.
2.  **Delegación al Planificador de Mantenimiento:** Se le solicitó evaluar la criticidad y consultar stock. Determinó criticidad **ALTA** (aunque la falla fue en tierra, la afectación a la barra esencial y el antecedente de la flota mantienen el nivel máximo). Consultó el stock y reportó **0 unidades** del repuesto TR2, con un plazo de reposición de **15 días** y sin alternativa técnica disponible.
3.  **Delegación al Coordinador de Mantenimiento:** Se le solicitó crear la OT y gestionar las comunicaciones. Creó la **OT-20261023-003** con criticidad ALTA y estado "PENDIENTE_APROBACION_SUPERVISOR". Envío el correo a `supervision.mantenimiento@taller-sh32.cl` solicitando la aprobación obligatoria de la OT crítica (política 2: nadie del equipo aprueba OT críticas). Envío el correo a `abastecimiento@taller-sh32.cl` informando la falta de stock, el plazo de 15 días y la inexistencia de alternativa (política 3).

**Próximos pasos**
1.  **Aprobación de Supervisión:** Esperar la aprobación de la OT-20261023-003 por parte de la supervisión de mantenimiento. La aeronave queda **NO apta para vuelo (AOG)** hasta que se apruebe y ejecute la reparación.
2.  **Reposición del Repuesto:** Seguir el proceso de compra del TR2 con abastecimiento, considerando los 15 días de plazo. Evaluar si es posible un préstamo desde otra base o flota para reducir el tiempo de inactividad.
3.  **Ejecución de la Reparación:** Una vez aprobado y con el repuesto disponible, realizar el reemplazo del

In [ ]:
pausa()
# Escenario D: pregunta sin relación con el manual
traza_D = sistema.atender("¿Cuántos perros tienen el pelaje blanco?")

Pausa de 30 segundos (límite de tokens por minuto).


### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| - | - | (sin herramientas) | - |

**Delegaciones del Manager:** ninguna  
**Agentes que usaron herramientas:** ninguno (respondió el Manager)  
**OT creadas:** ninguna  
**Correos:** ninguno  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 5297 · **Llamadas al modelo:** 2 · **Tiempo:** 37.7 s

---
### Respuesta del equipo
**Diagnóstico**
La solicitud recibida pregunta por la cantidad de perros con pelaje blanco, un tema completamente ajeno al sistema eléctrico de la flota SH-32 y fuera del alcance de las competencias del taller de mantenimiento eléctrico. No corresponde a un reporte de falla, una consulta técnica de la aeronave ni un seguimiento de una OT existente.

**Decisiones tomadas**
Se aplicó la política obligatoria de declinación amable para temas ajenos al sistema eléctrico SH-32. No se delegaron tareas al Técnico Diagnosticador, al Planificador de Mantenimiento ni al Coordinador de Mantenimiento, ya que la consulta no involucra componentes, sistemas eléctricos, criticidad ni stock de repuestos de la flota. No se crearon órdenes de trabajo ni se consultaron manuales técnicos.

**Próximos pasos**
No hay próximos pasos técnicos. Se invita al solicitante a redirigir su consulta al área correspondiente o a presentar una nueva solicitud relacionada con el mantenimiento eléctrico de la flota SH-32.

**Fuentes**
No se consultaron páginas del manual ni se generaron números de OT para esta solicitud.

In [ ]:
pausa()
# Escenario E: seguimiento con la memoria de la sesión
traza_E = sistema.atender("¿Qué número de OT abriste para el caso del TR2 y en qué estado quedó?")

Pausa de 30 segundos (límite de tokens por minuto).


### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| - | - | (sin herramientas) | - |

**Delegaciones del Manager:** ninguna  
**Agentes que usaron herramientas:** ninguno (respondió el Manager)  
**OT creadas:** ninguna  
**Correos:** ninguno  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 3091 · **Llamadas al modelo:** 1 · **Tiempo:** 2.6 s

---
### Respuesta del equipo
**Diagnóstico**
La falla reportada en la aeronave SH-32-03 corresponde a la ausencia de tensión de 28 V en la salida del transformador-rectificador TR2, lo que deja la barra esencial PP2 con tensión baja. El TR2 forma parte de la red trifásica 2 y es responsable de alimentar la barra esencial PP2 de 28 V, acoplada a la batería y a circuitos vitales. La falla coincide con el escenario de "pérdida de un transformador-rectificador" descrito en el manual de mantenimiento eléctrico SH-32. Existe un antecedente directo: la aeronave SH-32-02 presentó la misma falla el 02/05/2026 (OT-20260502-004), donde se reemplazó el TR2.

**Decisiones tomadas**
Se aplicó la política de seguimiento de conversación previa (política 6): la consulta es un seguimiento de un caso ya atendido en esta misma sesión, por lo que no se delegaron tareas nuevas al Técnico Diagnosticador, al Planificador de Mantenimiento ni al Coordinador de Mantenimiento. La información se extrajo directamente de la conversación reciente y de la memoria de largo plazo. No se creó ninguna OT nueva.

**Próximos pasos**
- La OT-20261023-003 se encuentra en estado **PENDIENTE_APROBACION_SUPERVISOR**. Dado que la criticidad es ALTA, el Coordinador de Mantenimiento envió el correo a supervision.mantenimiento@taller-sh32.cl solicitando la aprobación. Nadie del equipo técnico aprueba OT críticas; solo una persona de supervisión puede hacerlo.
- Una vez aprobada, se procederá con el reemplazo del Transformador-Rectificador TR2.
- La aeronave SH-32-03 queda **NO apta para vuelo (AOG)** hasta completar la reparación y la verificación.
- Verificar stock del repuesto TR2; si no hay stock, el Coordinador enviará correo a abastecimiento@taller-sh32.cl con el plazo de reposición.

**Fuentes**
- Manual de mantenimiento eléctrico SH-32: páginas 4, 6 y 19 (descripción del TR2, red trifásica 2, barra esencial PP2 de 28 V y escenario de pérdida de transformador-rectificador).
- OT-20261023-003 (TR2, SH-32-03, 23/10/2026, estado: PENDIENTE_APROBACION_SUPERVISOR).
- OT-20260502-004 (TR2, SH-32-02, 02/05/2026, antecedente de reempl

### 6.1 Resumen de los escenarios

In [ ]:
# Resumen de los escenarios que se ejecutaron
trazas = {e: globals()[f"traza_{e}"] for e in "ABCDE" if f"traza_{e}" in globals()}
if len(trazas) < 5:
    print("Escenarios sin ejecutar:", ", ".join(e for e in "ABCDE" if e not in trazas))
resumen = pd.DataFrame([{
    "escenario": e,
    "secuencia real": " → ".join(t["secuencia"]) or "-",
    "delegaciones": len(t["delegaciones"]),
    "agentes": " → ".join(a.split()[0] for a in t["agentes"]) or "Manager",
    "OT (estado)": ", ".join(f"{o['id']} ({o['estado']})" for o in t["ots_creadas"]) or "-",
    "compuerta ajustó": "sí" if any(o["ajustada_por_compuerta"] for o in t["ots_creadas"]) else "-",
    "correos": ", ".join(c["para"].split("@")[0] for c in t["correos"]) or "-",
    "auditoría": ", ".join(t["correcciones_auditoria"]) or "-",
    "tokens": t["tokens"], "segundos": t["segundos"],
    "error": (t["error"] or "")[:60]} for e, t in trazas.items()])
display(resumen)
resumen.to_csv(DIR_EVIDENCIA / "resumen_escenarios.csv", index=False)

print("\nÓrdenes de trabajo creadas:")
display(pd.DataFrame(_leer_json(ARCH_OT))[["id", "aeronave", "componente", "criticidad", "estado"]]
        if _leer_json(ARCH_OT) else "(ninguna)")
print("\nCorreos enviados:")
display(pd.DataFrame(_leer_json(ARCH_CORREOS))[["timestamp", "para", "asunto"]]
        if _leer_json(ARCH_CORREOS) else "(ninguno)")

ENTORNO["inventario"]["TR2"]["stock"] = 1   # vuelve el stock original

,escenario,secuencia real,delegaciones,agentes,OT (estado),compuerta ajustó,correos,auditoría,tokens,segundos,error
0,A,buscar_manual → buscar_manual → buscar_histori...,4,Técnico → Planificador → Coordinador,OT-20261023-001 (ABIERTA),-,-,-,17463,394.2,
1,B,buscar_manual → buscar_historial_fallas → busc...,3,Técnico → Planificador → Coordinador,OT-20261023-002 (PENDIENTE_APROBACION_SUPERVISOR),-,supervision.mantenimiento,-,15063,322.4,
2,C,buscar_manual → buscar_manual → buscar_manual ...,3,Técnico → Planificador → Coordinador,OT-20261023-003 (PENDIENTE_APROBACION_SUPERVISOR),-,"supervision.mantenimiento, abastecimiento",-,16066,385.0,
3,D,-,0,Manager,-,-,-,-,5297,37.7,
4,E,-,0,Manager,-,-,-,-,3091,2.6,



Órdenes de trabajo creadas:


,id,aeronave,componente,criticidad,estado
0,OT-20261023-001,SH-32-01,49P,MEDIA,ABIERTA
1,OT-20261023-002,SH-32-02,49P,ALTA,PENDIENTE_APROBACION_SUPERVISOR
2,OT-20261023-003,SH-32-03,TR2,ALTA,PENDIENTE_APROBACION_SUPERVISOR



Correos enviados:


,timestamp,para,asunto
0,23/10/2026 04:08:51,supervision.mantenimiento@taller-sh32.cl,Solicitud de aprobación OT-20261023-002 - SH-3...
1,23/10/2026 04:15:49,supervision.mantenimiento@taller-sh32.cl,APROBACIÓN OT CRÍTICA - SH-32-03 - TR2 - AOG
2,23/10/2026 04:16:08,abastecimiento@taller-sh32.cl,FALTA DE STOCK - Repuesto TR2 - SH-32-03 - Pla...


## 7. Continuidad entre sesiones

Primero se muestra lo que guardó la memoria de la sesión. Después se crea un sistema nuevo, con la memoria de corto plazo vacía, como si empezara el turno de noche (escenario F). Para responder por el TR2 tiene que usar la memoria de largo plazo.

In [ ]:
print(f"Turnos en memoria de corto plazo: {len(sistema.memoria_cp)} (ventana de {sistema.memoria_cp.ventana})\n")
print(sistema.memoria_cp.como_texto())

Turnos en memoria de corto plazo: 5 (ventana de 6)

[04:03] Técnico: Reporto falla en la aeronave SH-32-01, en tierra: durante la prueba en plataforma se enciende la luz de transferencia de la red 1. Sospechamos del relé 49P.
[04:03] Sistema: [OT: OT-20261023-001 49P ABIERTA] **Diagnóstico**
La falla reportada en la aeronave SH-32-01 corresponde a la iluminación de la luz de transferencia de la red 1 durante una prueba en plataforma (en tierra). El componente identificado es el relé de transferencia 49P, perteneciente al sistema de transferencia automática de la red eléctrica. Existe una coincidencia directa con el historial de la misma aeronave (OT-20260314-001, 14/03/2026), donde se reemplazó este mismo relé por una falla similar detectada en vuelo. El diagnóstico técnico confirma la sospecha inicial del equipo de taller.

**Deci
[04:09] Técnico: Reporto falla en la aeronave SH-32-02: la luz de transferencia de la red 1 se encendió EN VUELO y la tripulación informó pérdida de tensión

In [ ]:
pausa()
sistema_turno2 = SistemaJerarquico("sesion-2")   # sesión nueva, sin memoria de corto plazo
print("Memoria de corto plazo de la sesión 2:", sistema_turno2.memoria_cp.como_texto(), "\n")
traza_F = sistema_turno2.atender(
    "Soy del turno de noche. ¿El TR2 ha fallado antes en la flota? ¿Qué se hizo y hay alguna OT pendiente?")

ots_tr2 = [o["id"] for o in _leer_json(ARCH_OT) if o["componente"] == "TR2"]
recordo = any(i in traza_F["respuesta"] for i in ots_tr2)
print("La sesión 2 menciona la OT del TR2 creada en la sesión 1:" , "sí" if recordo else "no", ots_tr2)

Pausa de 30 segundos (límite de tokens por minuto).
Memoria de corto plazo de la sesión 2: (sin turnos previos en esta sesión) 



### Lo que decidió el Manager
| # | Agente | Herramienta | Hora |
|---|---|---|---|
| 1 | Técnico Diagnosticador | `buscar_historial_fallas` | 04:19:27 |
| 2 | Técnico Diagnosticador | `buscar_manual` | 04:19:36 |

**Delegaciones del Manager:** Técnico Diagnosticador  
**Agentes que usaron herramientas:** Técnico Diagnosticador  
**OT creadas:** ninguna  
**Correos:** ninguno  
**Correcciones de la auditoría:** ninguna  
**Tokens:** 10374 · **Llamadas al modelo:** 3 · **Tiempo:** 164.8 s

---
### Respuesta del equipo
**Diagnóstico**
El componente TR2 (Transformador-Rectificador) ha presentado fallas previas en la flota SH-32. Se identifican dos casos documentados:
1. **SH-32-02 (02/05/2026)**: Falla por ausencia de salida de 28 V, dejando la barra esencial PP2 alimentada solo por batería. Se realizó el reemplazo del TR2 bajo la OT-20260502-004, la cual fue aprobada por supervisión y ejecutada.
2. **SH-32-03 (23/10/2026)**: Falla por no entrega de 28 V y tensión baja en la barra. La aeronave se encuentra AOG (No apta para vuelo). La acción planificada es el reemplazo del TR2 bajo la OT-20261023-003, la cual se encuentra **PENDIENTE DE APROBACIÓN DE SUPERVISIÓN**.

**Decisiones tomadas**
- Se delegó la verificación técnica e histórica al **Técnico Diagnosticador** para confirmar los antecedentes de fallas del TR2, las acciones correctivas realizadas y el estado actual de las órdenes de trabajo (OT) asociadas, citando las referencias del manual.
- **No se delegó al Planificador de Mantenimiento ni al Coordinador de Mantenimiento** porque la solicitud es una consulta de información técnica e histórica (política 4 y 6), no el reporte de una falla nueva que requiera evaluación de criticidad, consulta de stock o creación de una nueva OT.
- Se respetó la política de que las OT críticas (ALTA) requieren aprobación de una persona de supervisión, por lo que la OT-20261023-003 permanece en espera de dicha aprobación y no se ha ejecutado.

**Próximos pasos**
1. **Aprobación de OT-20261023-003**: El supervisor de mantenimiento debe revisar y aprobar la OT pendiente para la aeronave SH-32-03, permitiendo así la ejecución del reemplazo del TR2.
2. **Ejecución del reemplazo**: Una vez aprobada la OT, el equipo técnico procederá con el reemplazo del TR2 en la SH-32-03 para restaurar la alimentación de 28 V a la barra esencial PP2 y habilitar la aeronave para vuelo.
3. **Cierre de OT**: Tras la verificación post-mantenimiento, se cerrará la OT-20261023-003 y se actualizará el historial de la flota.

**Fuentes**
- **Manual de Mantenimiento Eléctrico SH-32**: Secciones correspondientes al sistema de Red Trifásica 2 y Barra Esencial PP2 de 28 V (referencia técnica del componente TR2).
- **OT-20260502-004**: Orden de trabajo cerrada para SH-32-02 (reemplazo de TR2, aprobada y ejecutada).
- **OT-20261023-003**: Orden de trabajo pendiente de aprobación para SH-32-03 (reemplazo de TR2, criticidad ALTA).

La sesión 2 menciona la OT del TR2 creada en la sesión 1: sí ['OT-20261023-003']


## 8. Evidencias para el repositorio

Guarda en `evidencias/` las órdenes, los correos, los registros, el resumen de los escenarios, las pruebas, la tabla del umbral y la memoria, y descarga todo en un `.zip`.

In [ ]:
DIAGRAMA_MERMAID = """```mermaid
flowchart TD
    U[Persona técnica] -->|solicitud| M[Manager · Jefe de Taller<br/>Process.hierarchical<br/>planifica y delega]
    CP[(Memoria corto plazo<br/>ventana 6 turnos)] -->|historial| M
    LP[(Memoria largo plazo<br/>FAISS episodios)] -->|contexto semántico| M
    M -->|delega| D[Técnico Diagnosticador]
    M -->|delega| P[Planificador de Mantenimiento]
    M -->|delega| C[Coordinador de Mantenimiento]
    D --> T1[buscar_manual<br/>RAG manual SH-32]
    D --> T2[buscar_historial_fallas]
    P --> T3{evaluar_criticidad}
    P --> T4[consultar_stock]
    C --> T5[crear_orden_trabajo]
    C --> T6[enviar_correo]
    T5 --> G{Compuerta en código}
    G -->|ALTA| PA[OT pendiente aprobación<br/>+ correo supervisión]
    G -->|sin stock| ES[OT en espera de repuesto<br/>+ correo abastecimiento]
    G -->|MEDIA/BAJA con stock| AB[OT abierta]
    T5 --> LP
    T2 --> LP
    M --> AU[Auditoría de políticas<br/>+ traza JSONL]
    M -->|respuesta con fuentes| U
```"""
(DIR_EVIDENCIA / "diagrama_orquestacion.md").write_text(DIAGRAMA_MERMAID, encoding="utf-8")
shutil.copy(memoria_lp.registro, DIR_EVIDENCIA / "memoria_largo_plazo_episodios.jsonl")

zip_path = shutil.make_archive(str(BASE / "evidencias_EP2"), "zip", DIR_EVIDENCIA)
print("Evidencias:", sorted(p.name for p in DIR_EVIDENCIA.iterdir()))
if EN_COLAB:
    files.download(zip_path)

Evidencias: ['calibracion_umbral.csv', 'correos_enviados.json', 'diagrama_orquestacion.md', 'memoria_largo_plazo_episodios.jsonl', 'ordenes_trabajo.json', 'pruebas_herramientas.csv', 'resumen_escenarios.csv', 'trazas_agente.jsonl']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Uso libre para la demo

Quita los `#` para conversar con el equipo de agentes. Escribe `salir` para terminar.

In [ ]:
# while True:
#     q = input("Tú: ")
#     if q.strip().lower() in ("salir", "exit"):
#         break
#     sistema.atender(q)

## Conclusiones y reflexiones individuales

[Conclusiones del equipo]

**Diego Cabrera:** [reflexión]

**Luis Muñoz:** [reflexión]

**Aran Opazo:** [reflexión]

**Eric Rubilar:** [reflexión]

**Uso de IA:** [herramienta, para qué se usó y cómo se revisó]